In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Lodhi_Road_Delhi_IITM_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,NaN,NaN,89.0,288.0,NaN,116.0,NaN,NaN,NaN,133.0,296.0,218.0
1,2,NaN,NaN,NaN,202.0,211.0,129.0,NaN,NaN,NaN,NaN,246.0,219.0
2,3,NaN,NaN,95.0,258.0,183.0,97.0,NaN,NaN,NaN,NaN,325.0,238.0
3,4,NaN,NaN,90.0,314.0,168.0,140.0,NaN,NaN,NaN,NaN,304.0,171.0
4,5,NaN,98.0,83.0,329.0,155.0,83.0,NaN,NaN,NaN,45.0,307.0,127.0
5,6,NaN,98.0,82.0,208.0,111.0,98.0,NaN,NaN,NaN,NaN,249.0,165.0
6,7,NaN,118.0,109.0,NaN,161.0,153.0,NaN,NaN,NaN,NaN,177.0,151.0
7,8,NaN,NaN,NaN,295.0,127.0,121.0,NaN,NaN,NaN,NaN,NaN,189.0
8,9,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,133.0,296.0,113.0
9,10,NaN,176.0,NaN,310.0,107.0,NaN,NaN,NaN,NaN,NaN,NaN,159.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,147.052632,114.625,89.000000,175.36,112.9375,82.75,15.5,15.5,101.153846,134.75,296.000000,218.000000
1,2,147.052632,114.625,136.387097,202.00,112.9375,82.75,15.5,15.5,101.153846,134.75,246.000000,219.000000
2,3,147.052632,114.625,95.000000,175.36,183.0000,82.75,15.5,15.5,101.153846,134.75,325.000000,238.000000
3,4,147.052632,114.625,90.000000,175.36,168.0000,82.75,15.5,15.5,101.153846,134.75,304.000000,171.000000
4,5,147.052632,98.000,83.000000,175.36,155.0000,82.75,15.5,15.5,101.153846,134.75,307.000000,127.000000
5,6,147.052632,98.000,82.000000,208.00,111.0000,82.75,15.5,15.5,101.153846,134.75,249.000000,165.000000
6,7,147.052632,118.000,109.000000,175.36,161.0000,82.75,15.5,15.5,101.153846,134.75,177.000000,151.000000
7,8,147.052632,114.625,136.387097,175.36,127.0000,82.75,15.5,15.5,101.153846,134.75,218.392857,189.000000
8,9,147.052632,114.625,136.387097,175.36,112.0000,82.75,15.5,15.5,101.153846,134.75,296.000000,113.000000
9,10,147.052632,114.625,136.387097,175.36,107.0000,82.75,15.5,15.5,101.153846,134.75,218.392857,159.000000
